In [10]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score, confusion_matrix
data = pd.read_csv('weather.csv')

data['date'] = pd. to_datetime (data['date'])
data['month'] = data['date'].dt.month
data['day_of_year'] = data[ 'date'].dt.dayofyear

def categorize_weather(row):
  if row['prcp'] > 0: 
    return 'Rainy'
  elif row['tavg'] > 20 and row['prcp'] == 0:
    return 'Sunny'
  else:
    return 'Cloudy'
data['weather_condition'] = data.apply(categorize_weather, axis=1)

X = data[['tavg', 'tmin', 'tmax', 'prcp', 'wdir', 'wspd', 'wpgt', 'pres', 'month', 'day_of_year']]
y = data['weather_condition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
priors = y_train.value_counts(normalize=True).to_dict()



In [11]:
likelihoods = {}
for condition in y_train.unique():
  subset = X_train[y_train == condition]
  likelihoods[condition] = {
    'mean':subset.mean(),
    'std': subset.std()
  }

In [12]:
def gaussian_pdf(x, mean, std):
  return (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(-0.5 * ((x - mean) / std) ** 2)
def predict(X):
  predictions = []
  for _, row in X.iterrows():
    posteriors = {}
    for condition in priors.keys():
      prior = priors [condition]
      likelihood = 1
      for feature in X.columns:
        mean = likelihoods[condition]['mean'][feature]
        std = likelihoods[condition]['std'][feature]
        likelihood *= gaussian_pdf(row[feature], mean, std)
      posteriors[condition] = prior * likelihood
    predictions.append(max(posteriors, key=posteriors.get))
  return predictions

In [14]:
y_pred = predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print (f"Model Accuracy: {accuracy:.2f}")
con_matrix = confusion_matrix(y_test, y_pred, labels=['Sunny', 'Rainy', 'Cloudy'])
print ("Confusion Matrix:")
print(con_matrix)

Model Accuracy: 0.62
Confusion Matrix:
[[ 0  0 11]
 [ 0  0 17]
 [ 0  0 46]]


/var/folders/f_/m4zv4k0d55z9c9fwsq0_hyrr0000gn/T/ipykernel_83415/1352786938.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  return (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(-0.5 * ((x - mean) / std) ** 2)
/var/folders/f_/m4zv4k0d55z9c9fwsq0_hyrr0000gn/T/ipykernel_83415/1352786938.py:2: RuntimeWarning: invalid value encountered in scalar multiply
  return (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(-0.5 * ((x - mean) / std) ** 2)
/var/folders/f_/m4zv4k0d55z9c9fwsq0_hyrr0000gn/T/ipykernel_83415/1352786938.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(-0.5 * ((x - mean) / std) ** 2)


In [15]:
sample_date = {'tavg': 15, 'tmin': 10, 'tmax': 20, 'prcp': 0, 'wdir': 180, 'wspd': 10, 'wpgt': 15, 'pres': 1015, 'month': 6, 'day_of_year': 180}
sample_df = pd.DataFrame([sample_date])
predicted_condition = predict(sample_df)
print(f"Predicted weather condition: {predicted_condition[0]}")

Predicted weather condition: Cloudy


/var/folders/f_/m4zv4k0d55z9c9fwsq0_hyrr0000gn/T/ipykernel_83415/1352786938.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  return (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(-0.5 * ((x - mean) / std) ** 2)
/var/folders/f_/m4zv4k0d55z9c9fwsq0_hyrr0000gn/T/ipykernel_83415/1352786938.py:2: RuntimeWarning: invalid value encountered in scalar divide
  return (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(-0.5 * ((x - mean) / std) ** 2)
